# 训推共卡框架运行切换演示案例
在强化学习（RL）中，为了提升资源利用率，通常会采用训练与推理共卡模式（Col-locate mode）。在该模式下，训练和推理会交替切换。通过Ray进行资源调度，以Megatron、Torch作为训练框架，SGLang作为推理框架，可以逐步理解训推共卡时的运行原理。

|案例   |简介   |
|:-------------|:------------|
|case1: ray_placement_group| ray资源调度中预留资源的使用 |
|case2: ray_sglang| ray调度SGLang运行案例 |
|case3: ray_sglang_multi_node| 多机运行SGLang推理 |
|case4: ray_megatron | ray调度megatron运行 |
|case5: offload_training | 普通模式的训练显存资源卸载 |
|case6: torch_memory_saver| CUDA Hook方式卸载训练资源 |
|case7: train_infer_interleave| 训练推理交替使用资源 |


[**相关文章 Link**](https://zhuanlan.zhihu.com/p/2028552054742763264)

Author: kaiyuan

Email: kaiyuanxie@yeah.net

## 运行环境

需要镜像里面包含Megatron、SGLang、Ray等框架，采用如下镜像：

```
docker pull slimerl/slime:latest
```
启动示例：
```
docker run -itd --rm \
  --gpus all \
  --network=host \
  --ipc=host \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  -v /data/nfs_kaiyuan:/data/nfs_kaiyuan \
  --name ray_pg_demo \
  slimerl/slime:latest bash
```

本示例中镜像中的版本：
```
torch                      2.9.1+cu129
torch_c_dlpack_ext               0.1.5
torch_memory_saver               0.0.9
torchao                     0.9.0
torchaudio                    2.9.1+cu129
torchcodec                    0.8.0
torchvision                   0.24.1+cu129
transformer_engine_torch            2.10.0
megatron-bridge                 0.3.0rc0
megatron-core                  0.16.0rc0        /root/Megatron-LM
ray                       2.54.0
sglang                      0.5.9            /sgl-workspace/sglang/python
sglang-router                   0.3.2
```


## Case1: Ray的预占(Placement Group)资源调度

Ray Placement Group能够预占资源，然后再次分配。
`PlacementGroup`里的`strategy`主要决定bundle在节点上的分布方式：

- `PACK`：尽量把bundle塞到尽可能少的节点上（偏“聚拢”），放不下再扩到别的节点。
- `SPREAD`：尽量把bundle分散到不同节点上（偏“分散”），但不是强约束，资源紧张时可能还是有多个bundle落同一节点。
- `STRICT_PACK`：强制所有bundle必须在同一节点；若单节点放不下，PG创建直接失败。
- `STRICT_SPREAD`：强制每个bundle分布到不同节点；若节点数不够或资源不满足，PG创建直接失败。

简单选型：
- 想要跨机容错/带宽均衡：优先`SPREAD`或`STRICT_SPREAD`
- 想要单机内高速通信：优先`PACK`或`STRICT_PACK`
- 需要“必须满足拓扑约束，否则宁可失败”：用`STRICT_*`  
- 只想“尽量满足，资源不够也先跑起来”：用非`STRICT`的`PACK/SPREAD`

示例：PlacementGroup对资源调度的控制

Ray+PlacementGroup的小示例：先申请N个GPUbundle，再逐个探测每个bundle实际落到哪台机器和哪张物理GPU，然后按固定规则重排bundle顺序，最后用重排后的bundle索引启动worker，验证logical_slot到physical_gpu的映射可控且可复现。

单机快速验证

执行：
```
python ray_bundle_reorder_demo.py --num-gpus 8 --strategy PACK --reorder-mode reverse
```
这会最直观看到raw mapping与reordered差异。


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import argparse

import ray
from ray.util.placement_group import placement_group, remove_placement_group
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy


@ray.remote(num_gpus=1)
class Probe:
    def probe(self):
        import socket

        gpu_ids = ray.get_gpu_ids()
        return {
            "node": socket.gethostname(),
            "gpu_id": int(gpu_ids[0]) if gpu_ids else -1,
            "all_gpu_ids": gpu_ids,
        }


def sort_key(item):
    # item = (bundle_idx, node, gpu_id)
    bundle_idx, node, gpu_id = item
    try:
        node_ip = tuple(int(x) for x in node.split("."))
        return (0, node_ip, gpu_id, bundle_idx)
    except Exception:
        return (1, node, gpu_id, bundle_idx)


def probe_bundle_mapping(pg, num_bundles):
    probes = []
    for i in range(num_bundles):
        actor = Probe.options(
            scheduling_strategy=PlacementGroupSchedulingStrategy(
                placement_group=pg,
                placement_group_bundle_index=i,
                placement_group_capture_child_tasks=True,
            )
        ).remote()
        probes.append(actor)

    infos = ray.get([p.probe.remote() for p in probes])
    for p in probes:
        ray.kill(p)

    bundle_infos = [(i, infos[i]["node"], infos[i]["gpu_id"]) for i in range(num_bundles)]
    return infos, bundle_infos


def build_reordered(bundle_infos, reorder_mode):
    if reorder_mode == "node_gpu":
        return [x[0] for x in sorted(bundle_infos, key=sort_key)]
    if reorder_mode == "reverse":
        return [x[0] for x in reversed(bundle_infos)]
    if reorder_mode == "odd_even":
        odds = [x[0] for x in bundle_infos if x[0] % 2 == 1]
        evens = [x[0] for x in bundle_infos if x[0] % 2 == 0]
        return odds + evens
    raise ValueError(f"Unknown reorder_mode={reorder_mode}")


def print_diff_summary(bundle_infos, reordered):
    # old_slot == bundle_idx in this demo.
    new_slot_of_bundle = {bundle_idx: i for i, bundle_idx in enumerate(reordered)}
    moved = 0
    print("\n[slot diff summary]")
    print("bundle | old_slot -> new_slot | changed")
    print("----------------------------------------")
    for bundle_idx, _, _ in bundle_infos:
        old_slot = bundle_idx
        new_slot = new_slot_of_bundle[bundle_idx]
        changed = old_slot != new_slot
        if changed:
            moved += 1
        print(
            f"{bundle_idx:>6} | {old_slot:>8} -> {new_slot:<8} | "
            f"{'yes' if changed else 'no'}"
        )
    print(f"moved bundles: {moved}/{len(bundle_infos)}")


def run_demo(num_gpus, strategy, reorder_mode):
    total_gpus = int(ray.cluster_resources().get("GPU", 0))
    if total_gpus < num_gpus:
        raise RuntimeError(f"Need {num_gpus} GPUs, found {total_gpus}.")

    pg = placement_group([{"CPU": 1, "GPU": 1} for _ in range(num_gpus)], strategy=strategy)
    print(f"waiting placement group ready... (strategy={strategy}, num_gpus={num_gpus})")
    ray.get(pg.ready())
    print("placement group ready")

    infos, bundle_infos = probe_bundle_mapping(pg, num_gpus)
    reordered = build_reordered(bundle_infos, reorder_mode)

    print("\n[raw bundle mapping]")
    for b, node, gid in bundle_infos:
        print(f"bundle={b} -> node={node}, physical_gpu={gid}, ray_gpu_ids={infos[b]['all_gpu_ids']}")

    print("\n[reordered logical slots]")
    for logical_slot, bundle_idx in enumerate(reordered):
        node = infos[bundle_idx]["node"]
        gid = infos[bundle_idx]["gpu_id"]
        print(f"logical_slot={logical_slot} -> bundle={bundle_idx} -> node={node}, physical_gpu={gid}")
    print_diff_summary(bundle_infos, reordered)

    # Optional: show how to launch workers with reordered bundle indices.
    workers = []
    for logical_slot, bundle_idx in enumerate(reordered):
        actor = Probe.options(
            scheduling_strategy=PlacementGroupSchedulingStrategy(
                placement_group=pg,
                placement_group_bundle_index=bundle_idx,
                placement_group_capture_child_tasks=True,
            )
        ).remote()
        workers.append((logical_slot, actor))
    worker_infos = [(slot, ray.get(a.probe.remote())) for slot, a in workers]
    for _, a in workers:
        ray.kill(a)

    print("\n[workers launched with reordered bundle indices]")
    for slot, info in worker_infos:
        print(
            f"logical_slot={slot} runs on node={info['node']}, "
            f"physical_gpu={info['gpu_id']}, ray_gpu_ids={info['all_gpu_ids']}"
        )

    remove_placement_group(pg)
    print("\ndone.")


def parse_args():
    parser = argparse.ArgumentParser(description="Standalone bundle->physical GPU reorder demo")
    parser.add_argument("--address", type=str, default=None, help="Ray address, e.g. auto")
    parser.add_argument("--num-cpus", type=int, default=8, help="Used when starting local ray")
    parser.add_argument("--num-gpus", type=int, default=8, help="Number of bundles to request")
    parser.add_argument("--strategy", type=str, default="PACK", choices=["PACK", "SPREAD", "STRICT_PACK", "STRICT_SPREAD"])
    parser.add_argument(
        "--reorder-mode",
        type=str,
        default="node_gpu",
        choices=["node_gpu", "reverse", "odd_even"],
        help="How to reorder bundles; reverse/odd_even makes differences easier to observe",
    )
    return parser.parse_args()


def main():
    args = parse_args()
    init_kwargs = {"address": args.address} if args.address else {"num_cpus": args.num_cpus}
    ray.init(**init_kwargs)
    try:
        run_demo(args.num_gpus, args.strategy, args.reorder_mode)
    finally:
        ray.shutdown()


if __name__ == "__main__":
    main()


waiting placement group ready... (strategy=PACK, num_gpus=8)
placement group ready

[raw bundle mapping]
bundle=0 -> node=DevServer-BMS-292db97e, physical_gpu=0, ray_gpu_ids=[0]
bundle=1 -> node=DevServer-BMS-292db97e, physical_gpu=1, ray_gpu_ids=[1]
bundle=2 -> node=DevServer-BMS-292db97e, physical_gpu=2, ray_gpu_ids=[2]
bundle=3 -> node=DevServer-BMS-292db97e, physical_gpu=3, ray_gpu_ids=[3]
bundle=4 -> node=DevServer-BMS-292db97e, physical_gpu=4, ray_gpu_ids=[4]
bundle=5 -> node=DevServer-BMS-292db97e, physical_gpu=5, ray_gpu_ids=[5]
bundle=6 -> node=DevServer-BMS-292db97e, physical_gpu=6, ray_gpu_ids=[6]
bundle=7 -> node=DevServer-BMS-292db97e, physical_gpu=7, ray_gpu_ids=[7]

[reordered logical slots]
logical_slot=0 -> bundle=7 -> node=DevServer-BMS-292db97e, physical_gpu=7
logical_slot=1 -> bundle=6 -> node=DevServer-BMS-292db97e, physical_gpu=6
logical_slot=2 -> bundle=5 -> node=DevServer-BMS-292db97e, physical_gpu=5
logical_slot=3 -> bundle=4 -> node=DevServer-BMS-292db97e, phy

查看关键输出  
重点看三段：  
- `[raw bundle mapping]`：原始`bundle->node/gpu`映射  
- `[reordered logical slots]`：重排后`logical_slot->bundle`映射  
- `[slot diff summary]`：每个`bundle`是否移动以及`moved bundles`统计

`reorder`改变的是“逻辑槽位到物理资源”的对应关系，不是物理`GPU`位置本身。  
例如`reverse`下`logical_slot0`对应原来的`bundle7`，说明重排生效。

## Case2: Ray启动SGLang推理

Ray+PlacementGroup+SGLang

1.准备环境  
确保环境中可用`ray`和`sglang`，且至少有2张GPU。  
模型默认路径是`/data/nfs/kaiyuan/models/Qwen3-8B`。

2.启动方式  
单机本地运行：  
`python ray_sglang_pg_qwen3_8b_demo.py`  
连接已有Ray集群运行：  
`python ray_sglang_pg_qwen3_8b_demo.py --address auto`

3.执行流程  
- `ray.init()`初始化运行时。  
- 创建`placement_group([{"CPU":2,"GPU":2}],strategy="PACK")`预留2卡。  
- 用`PlacementGroupSchedulingStrategy`把`SGLangServerActor`固定到`bundle0`。  
- Actor内部启动SGLang服务，参数为`tp_size=2,nnodes=1,node_rank=0`。  
- 通过`/health_generate`轮询健康检查。  
- 默认模式下健康后会自动`stop`并退出；加`--keep-alive`则常驻。

4.常用参数  
- `--model-path`指定模型目录。  
- `--host`和`--port`指定服务监听地址与端口。  
- `--tp-size`指定张量并行大小（默认2）。  
- `--keep-alive`启动后不自动停止。

5.验证成功标志  
看到`start:`、`health:`、`server info:`输出且`health`为成功，即表示用例跑通。

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Ray+PlacementGroup+SGLang demo

目标：
1) 用placement_group预留2张GPU
2) 在该资源位上启动SGLang推理服务（tp-size=2）
3) 加载 Qwen3-8B（默认路径 /data/nfs/kaiyuan/models/Qwen3-8B）

使用方式：
如需保持服务不自动退出：
python ray_sglang_pg_qwen3_8b_demo.py --keep-alive
如需自动退出：
python ray_sglang_pg_qwen3_8b_demo.py
"""

import argparse
import os
import socket
import time
import multiprocessing

import ray
import requests
from ray.util.placement_group import placement_group, remove_placement_group
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy
from sglang.srt.server_args import ServerArgs
from sglang.srt.entrypoints.http_server import launch_server


def _wait_server_healthy(base_url, api_key, is_process_alive):
    headers = {
        "Content-Type": "application/json; charset=utf-8",
        "Authorization": f"Bearer {api_key}",
    }
    with requests.Session() as session:
        while True:
            try:
                response = session.get(f"{base_url}/health_generate", headers=headers, timeout=2)
                if response.status_code == 200:
                    return
            except requests.RequestException:
                pass

            if not is_process_alive():
                raise RuntimeError("Server process terminated unexpectedly.")

            time.sleep(2)


def launch_server_process_local(server_args: ServerArgs) -> multiprocessing.Process:

    multiprocessing.set_start_method("spawn", force=True)
    server_args.host = server_args.host.strip("[]")
    p = multiprocessing.Process(target=launch_server, args=(server_args,))
    p.start()
    _wait_server_healthy(
        base_url=server_args.url(),
        api_key=server_args.api_key,
        is_process_alive=lambda: p.is_alive(),
    )
    return p


@ray.remote(num_cpus=2, num_gpus=2)
class SGLangServerActor:
    def __init__(self, model_path: str, host: str, port: int, tp_size: int):
        self.model_path = model_path
        self.host = host
        self.port = port
        self.tp_size = tp_size
        self.proc = None

    def start(self):
        if self.proc is not None and self.proc.poll() is None:
            return {"status": "already_running", "pid": self.proc.pid}

        cvd = os.environ.get("CUDA_VISIBLE_DEVICES", "")
        server_args = ServerArgs(
            model_path=self.model_path,
            host=self.host,
            port=self.port,
            tp_size=self.tp_size,
            nnodes=1,
            node_rank=0,
            dist_init_addr=f"127.0.0.1:{self.port + 1}",
            nccl_port=self.port + 2,
            trust_remote_code=True,
        )
        # 通过launch_server_process风格启动sglangserver进程
        self.proc = launch_server_process_local(server_args)
        return {
            "status": "started",
            "pid": self.proc.pid,
            "cuda_visible_devices": cvd,
            "server_args": {
                "model_path": self.model_path,
                "host": self.host,
                "port": self.port,
                "tp_size": self.tp_size,
            },
            "node": socket.gethostname(),
        }

    def wait_until_healthy(self, timeout_s: int = 180):
        url = f"http://{self.host}:{self.port}/health_generate"
        deadline = time.time() + timeout_s
        last_error = None
        while time.time() < deadline:
            if self.proc is not None and self.proc.poll() is not None:
                raise RuntimeError(f"SGLang exited early, returncode={self.proc.returncode}")
            try:
                r = requests.get(url, timeout=2)
                if r.status_code == 200:
                    return {"status": "healthy", "url": url}
                last_error = f"status={r.status_code}"
            except Exception as e:
                last_error = str(e)
            time.sleep(2)
        raise TimeoutError(f"SGLang health check timeout: {last_error}")

    def info(self):
        return {
            "pid": self.proc.pid if self.proc else None,
            "alive": (self.proc is not None and self.proc.poll() is None),
            "host": self.host,
            "port": self.port,
            "model_path": self.model_path,
            "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
            "node": socket.gethostname(),
        }

    def stop(self):
        if self.proc is None:
            return {"status": "not_started"}
        if self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(timeout=15)
            except Exception:
                self.proc.kill()
        return {"status": "stopped", "returncode": self.proc.returncode}


def parse_args():
    parser = argparse.ArgumentParser(description="Ray placement + SGLang(Qwen3-8B) demo")
    parser.add_argument("--address", type=str, default=None, help="Ray address, e.g. auto")
    parser.add_argument("--num-cpus", type=int, default=8, help="Used when local ray.init()")
    parser.add_argument(
        "--model-path",
        type=str,
        default="/data/nfs/kaiyuan/models/Qwen3-8B",
        help="Qwen3-8B 路径",
    )
    parser.add_argument("--host", type=str, default="127.0.0.1", help="SGLang bind host")
    parser.add_argument("--port", type=int, default=30000, help="SGLang bind port")
    parser.add_argument("--tp-size", type=int, default=2, help="Tensor parallel size")
    parser.add_argument("--keep-alive", action="store_true", help="启动成功后不自动停止服务")
    return parser.parse_args()


def main():
    args = parse_args()
    init_kwargs = {"address": args.address} if args.address else {"num_cpus": args.num_cpus}
    ray.init(**init_kwargs)
    pg = None
    actor = None
    try:
        total_gpu = int(ray.cluster_resources().get("GPU", 0))
        if total_gpu < 2:
            raise RuntimeError(f"Need >=2 GPUs, found {total_gpu}")

        # 预留2卡资源
        pg = placement_group([{"CPU": 2, "GPU": 2}], strategy="PACK")
        print("waiting placement group ready...")
        ray.get(pg.ready())
        print("placement group ready.")

        # 固定调度到bundle0
        sched = PlacementGroupSchedulingStrategy(
            placement_group=pg,
            placement_group_bundle_index=0,
            placement_group_capture_child_tasks=True,
        )
        actor = SGLangServerActor.options(scheduling_strategy=sched).remote(
            model_path=args.model_path,
            host=args.host,
            port=args.port,
            tp_size=args.tp_size,
        )

        start_info = ray.get(actor.start.remote())
        print("start:", start_info)

        health = ray.get(actor.wait_until_healthy.remote())
        print("health:", health)
        print("server info:", ray.get(actor.info.remote()))

        if args.keep_alive:
            print("keep-alive enabled. Press Ctrl+C to exit.")
            while True:
                time.sleep(5)
        else:
            print("demo done, stopping server...")
            print("stop:", ray.get(actor.stop.remote()))
    finally:
        if actor is not None:
            try:
                ray.get(actor.stop.remote())
            except Exception:
                pass
        if pg is not None:
            remove_placement_group(pg)
        ray.shutdown()


if __name__ == "__main__":
    main()



## Case3 Ray调度多机SGLang推理


Ray+PlacementGroup+SGLang多机示例（2台机器，各1张卡，Qwen3-8B）

关键点：
1) 使用placement_group+STRICT_SPREAD，强制两个bundle分布到不同节点
2) 每个节点起一个SGLang子进程
3) 通过nnodes=2/node_rank/dist_init_addr组成一个tp_size=2的分布式推理服务


节点1（Head）
```
ray start \
  --head \
  --node-ip-address=xxx.xxx.xxx.xxx \
  --port=6379 \
  --dashboard-host=0.0.0.0 \
  --num-gpus=1
```
节点2（Worker）
```
ray start \
  --address='xxx.xxx.xxx.xxx:6379' \
  --num-gpus=1
```


任意节点上运行：

```
python ray_sglang_pg_qwen3_8b_multinode_demo.py --address='xxx.xxx.xxx.xxx:6379'
```

注意：
1. xxx.xxx.xxx.xxx需要替换为主节点的IP

2. 停止ray运行：
```
ray stop -f
```

3. 检查ray的状态：
```
python -c "import ray; ray.init(address='10.10.10.5:6379'); print(ray.cluster_resources())"
```


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-


import argparse
import multiprocessing
import os
import socket
import time

import ray
import requests
from ray.util.placement_group import placement_group, remove_placement_group
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy
from sglang.srt.entrypoints.http_server import launch_server
from sglang.srt.server_args import ServerArgs


def _wait_server_healthy(base_url, api_key, is_process_alive, timeout_s=240):
    headers = {
        "Content-Type": "application/json; charset=utf-8",
        "Authorization": f"Bearer {api_key}",
    }
    deadline = time.time() + timeout_s
    with requests.Session() as session:
        while time.time() < deadline:
            try:
                response = session.get(f"{base_url}/health_generate", headers=headers, timeout=2)
                if response.status_code == 200:
                    return
            except requests.RequestException:
                pass

            if not is_process_alive():
                raise RuntimeError("Server process terminated unexpectedly.")
            time.sleep(2)
    raise TimeoutError(f"Health check timeout for {base_url}")


def launch_server_process_local(server_args: ServerArgs) -> multiprocessing.Process:
    multiprocessing.set_start_method("spawn", force=True)
    server_args.host = server_args.host.strip("[]")
    p = multiprocessing.Process(target=launch_server, args=(server_args,))
    p.start()
    # 仅node_rank=0做健康检查即可，避免每个rank都阻塞在HTTP检查
    if getattr(server_args, "node_rank", 0) == 0:
        _wait_server_healthy(
            base_url=server_args.url(),
            api_key=server_args.api_key,
            is_process_alive=lambda: p.is_alive(),
        )
    return p


@ray.remote(num_cpus=2, num_gpus=1)
class SGLangNodeActor:
    def __init__(self):
        self.proc = None

    def get_node_ip(self):
        return ray.util.get_node_ip_address()

    def info(self):
        return {
            "hostname": socket.gethostname(),
            "node_ip": ray.util.get_node_ip_address(),
            "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
            "pid": self.proc.pid if self.proc else None,
            "alive": (self.proc is not None and self.proc.poll() is None),
        }

    def start(
        self,
        model_path: str,
        serve_host: str,
        serve_port: int,
        tp_size: int,
        nnodes: int,
        node_rank: int,
        dist_init_addr: str,
        nccl_port: int,
        gloo_ifname: str = None,
        nccl_ifname: str = None,
        startup_probe_seconds: int = 20,
    ):
        if self.proc is not None and self.proc.poll() is None:
            return {"status": "already_running", "pid": self.proc.pid, "node_rank": node_rank}

        # Ray actor再拉子进程时，显式透传网卡环境，避免和交互shell不一致。
        if gloo_ifname:
            os.environ["GLOO_SOCKET_IFNAME"] = gloo_ifname
        if nccl_ifname:
            os.environ["NCCL_SOCKET_IFNAME"] = nccl_ifname

        env_snapshot = {
            "MASTER_ADDR": dist_init_addr.split(":")[0],
            "MASTER_PORT": dist_init_addr.split(":")[1],
            "GLOO_SOCKET_IFNAME": os.environ.get("GLOO_SOCKET_IFNAME"),
            "NCCL_SOCKET_IFNAME": os.environ.get("NCCL_SOCKET_IFNAME"),
            "NCCL_DEBUG": os.environ.get("NCCL_DEBUG"),
            "TORCH_DISTRIBUTED_DEBUG": os.environ.get("TORCH_DISTRIBUTED_DEBUG"),
        }
        print(
            f"[SGLangNodeActor][rank={node_rank}] launch env: "
            f"host={socket.gethostname()} node_ip={ray.util.get_node_ip_address()} env={env_snapshot}"
        )

        server_args = ServerArgs(
            model_path=model_path,
            trust_remote_code=True,
            host=serve_host,
            port=serve_port,
            tp_size=tp_size,
            nnodes=nnodes,
            node_rank=node_rank,
            dist_init_addr=dist_init_addr,
            nccl_port=nccl_port,
        )
        self.proc = launch_server_process_local(server_args)

        # rank0已做HTTP健康检查；非rank0做短暂存活探测，尽早暴露初始化失败。
        if node_rank != 0 and startup_probe_seconds > 0:
            deadline = time.time() + startup_probe_seconds
            while time.time() < deadline:
                if self.proc.poll() is not None:
                    raise RuntimeError(
                        f"rank={node_rank} server process exited early, returncode={self.proc.returncode}"
                    )
                time.sleep(1)

        return {
            "status": "started",
            "node_rank": node_rank,
            "pid": self.proc.pid,
            "node_ip": ray.util.get_node_ip_address(),
            "hostname": socket.gethostname(),
            "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
            "gloo_ifname": os.environ.get("GLOO_SOCKET_IFNAME"),
            "nccl_ifname": os.environ.get("NCCL_SOCKET_IFNAME"),
            "server_url": f"http://{serve_host}:{serve_port}",
        }

    def stop(self):
        if self.proc is None:
            return {"status": "not_started"}
        if self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(timeout=15)
            except Exception:
                self.proc.kill()
        return {"status": "stopped", "returncode": self.proc.returncode}


def parse_args():
    parser = argparse.ArgumentParser(description="2-node A800 Ray+SGLang demo for Qwen3-8B")
    parser.add_argument("--address", type=str, default="auto", help="Ray 集群地址，默认 auto")
    parser.add_argument(
        "--model-path",
        type=str,
        default="/data/nfs/kaiyuan/models/Qwen3-8B",
        help="模型路径",
    )
    parser.add_argument("--serve-port", type=int, default=30000, help="每个节点上 SGLang 服务端口")
    parser.add_argument("--dist-init-port", type=int, default=31000, help="分布式初始化端口")
    parser.add_argument("--nccl-port", type=int, default=32000, help="NCCL 端口")
    parser.add_argument(
        "--gloo-ifname",
        type=str,
        default=None,
        help="可选：显式指定 GLOO_SOCKET_IFNAME（例如 ens8 / enp130s0f0）",
    )
    parser.add_argument(
        "--nccl-ifname",
        type=str,
        default=None,
        help="可选：显式指定 NCCL_SOCKET_IFNAME（例如 ens8 / enp130s0f0）",
    )
    parser.add_argument(
        "--startup-probe-seconds",
        type=int,
        default=20,
        help="非 rank0 启动后额外存活探测秒数",
    )
    parser.add_argument("--keep-alive", action="store_true", help="启动后保持服务")
    return parser.parse_args()


def _alive_gpu_nodes():
    gpu_nodes = []
    for n in ray.nodes():
        if not n.get("Alive", False):
            continue
        if float(n.get("Resources", {}).get("GPU", 0)) > 0:
            gpu_nodes.append(n)
    return gpu_nodes


def main():
    args = parse_args()
    ray.init(address=args.address)

    pg = None
    actors = []
    try:
        gpu_nodes = _alive_gpu_nodes()
        if len(gpu_nodes) < 2:
            raise RuntimeError(f"Need at least 2 alive GPU nodes, found {len(gpu_nodes)}.")

        # 两个bundle，严格分散到不同节点
        pg = placement_group([{"CPU": 2, "GPU": 1}, {"CPU": 2, "GPU": 1}], strategy="STRICT_SPREAD")
        print("waiting placement group ready (STRICT_SPREAD)...")
        ray.get(pg.ready())
        print("placement group ready.")

        for i in range(2):
            sched = PlacementGroupSchedulingStrategy(
                placement_group=pg,
                placement_group_bundle_index=i,
                placement_group_capture_child_tasks=True,
            )
            actor = SGLangNodeActor.options(scheduling_strategy=sched).remote()
            actors.append(actor)

        infos = ray.get([a.info.remote() for a in actors])
        print("node infos:", infos)

        master_ip = ray.get(actors[0].get_node_ip.remote())
        dist_init_addr = f"{master_ip}:{args.dist_init_port}"
        print("dist_init_addr:", dist_init_addr)

        start_refs = []
        for rank, actor in enumerate(actors):
            start_refs.append(
                actor.start.remote(
                    model_path=args.model_path,
                    serve_host="0.0.0.0",
                    serve_port=args.serve_port,
                    tp_size=2,
                    nnodes=2,
                    node_rank=rank,
                    dist_init_addr=dist_init_addr,
                    nccl_port=args.nccl_port,
                    gloo_ifname=args.gloo_ifname,
                    nccl_ifname=args.nccl_ifname,
                    startup_probe_seconds=args.startup_probe_seconds,
                )
            )
        start_infos = ray.get(start_refs)
        print("start infos:", start_infos)
        print(
            f"SGLang multi-node is up. Query endpoint: http://{master_ip}:{args.serve_port} "
            "(具体路由行为取决于 sglang 版本与配置)"
        )

        if args.keep_alive:
            print("keep-alive enabled. Press Ctrl+C to exit.")
            while True:
                time.sleep(5)
    finally:
        for a in actors:
            try:
                ray.get(a.stop.remote())
            except Exception:
                pass
        if pg is not None:
            remove_placement_group(pg)
        ray.shutdown()


if __name__ == "__main__":
    main()



## Case4 Ray调度megatron执行

Ray PG + Megatron training demo (single node, 8xA100)


默认路径：
- 模型: /data/nfs/kaiyuan/models/Qwen3-0.6B_torch_dist

说明：
- 这个脚本演示“如何用Ray placement_group预留8卡，再在该资源上启动Megatron训练进程”。
- 训练命令使用torchrun（单机 8 卡），你可以替换 --train-script和 --train-extra-args适配你的Megatron入口。

启动方式：python ray_pg_megatron_train_qwen06b_demo.py \
  --train-script /root/Megatron-LM/pretrain_gpt.py --use-mock-data

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-


import argparse
import os
import shlex
import socket
import subprocess
import time

import ray
from ray.util.placement_group import placement_group, remove_placement_group
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy

QWEN3_06B_MAX_POSITION_EMBEDDINGS = 32768
# Use a conservative seq length for OOM-safe demo.
QWEN3_06B_SEQ_LENGTH = 2048


@ray.remote(num_cpus=8, num_gpus=8)
class MegatronTrainActor:
    def __init__(self):
        self.proc = None

    def info(self):
        return {
            "node": socket.gethostname(),
            "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
            "pid": self.proc.pid if self.proc else None,
            "alive": (self.proc is not None and self.proc.poll() is None),
        }

    def start(
        self,
        train_script: str,
        model_path: str,
        data_path: str,
        output_dir: str,
        nproc_per_node: int,
        use_mock_data: bool,
        train_extra_args: list[str],
        env_overrides: dict[str, str],
    ):
        if self.proc is not None and self.proc.poll() is None:
            return {"status": "already_running", "pid": self.proc.pid}

        env = os.environ.copy()
        env.update(env_overrides or {})

        # For Qwen checkpoints converted to *_torch_dist, tokenizer files are usually in the paired HF dir.
        tokenizer_model_path = model_path[:-11] if model_path.endswith("_torch_dist") else model_path

        # Default MODEL_ARGS for Qwen3-0.6B (can still be overridden/extended via --train-extra-args)
        model_args = [
            "--swiglu",
            "--num-layers",
            "28",
            "--hidden-size",
            "1024",
            "--ffn-hidden-size",
            "3072",
            "--num-attention-heads",
            "16",
            "--group-query-attention",
            "--num-query-groups",
            "8",
            "--use-rotary-position-embeddings",
            "--disable-bias-linear",
            "--normalization",
            "RMSNorm",
            "--norm-epsilon",
            "1e-6",
            "--rotary-base",
            "1000000",
            "--vocab-size",
            "151936",
            "--kv-channels",
            "128",
            "--qk-layernorm",
            "--max-position-embeddings",
            str(QWEN3_06B_MAX_POSITION_EMBEDDINGS),
            "--seq-length",
            str(QWEN3_06B_SEQ_LENGTH),
            "--tokenizer-type",
            "HuggingFaceTokenizer",
            "--tokenizer-model",
            tokenizer_model_path,
        ]

        # 这是一个“可运行示例”的默认参数集合，具体参数可按你的Megatron脚本调整。
        cmd = [
            "torchrun",
            "--nproc_per_node",
            str(nproc_per_node),
            train_script,
            "--load",
            model_path,
            "--save",
            output_dir,
            "--micro-batch-size",
            "1",
            "--global-batch-size",
            "8",
            "--train-iters",
            "20",
            "--eval-interval",
            "20",
            "--eval-iters",
            "1",
            "--lr",
            "1e-6",
            "--save-interval",
            "20",
            "--tensor-model-parallel-size",
            "1",
            "--pipeline-model-parallel-size",
            "1",
            "--bf16",
            "--recompute-granularity",
            "full",
            "--recompute-method",
            "uniform",
            "--recompute-num-layers",
            "28",
        ]
        if use_mock_data:
            cmd += ["--mock-data"]
        else:
            cmd += ["--train-data-path", data_path]

        cmd = cmd + model_args + (train_extra_args or [])

        self.proc = subprocess.Popen(cmd, env=env)
        return {
            "status": "started",
            "pid": self.proc.pid,
            "node": socket.gethostname(),
            "cuda_visible_devices": env.get("CUDA_VISIBLE_DEVICES"),
            "cmd": " ".join(shlex.quote(x) for x in cmd),
        }

    def wait(self, timeout_s: int = 0):
        if self.proc is None:
            return {"status": "not_started"}
        if timeout_s and timeout_s > 0:
            try:
                code = self.proc.wait(timeout=timeout_s)
                return {"status": "finished", "returncode": code}
            except subprocess.TimeoutExpired:
                return {"status": "running", "pid": self.proc.pid}
        code = self.proc.wait()
        return {"status": "finished", "returncode": code}

    def stop(self):
        if self.proc is None:
            return {"status": "not_started"}
        if self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        return {"status": "stopped", "returncode": self.proc.returncode}


def parse_args():
    parser = argparse.ArgumentParser(description="Ray PG launch Megatron training on single-node 8 GPUs")
    parser.add_argument("--address", type=str, default=None, help="Ray address; empty = local mode")
    parser.add_argument("--num-cpus", type=int, default=16, help="CPUs for local ray.init()")
    parser.add_argument("--nproc-per-node", type=int, default=8, help="torchrun local world size")
    parser.add_argument(
        "--model-path",
        type=str,
        default="/data/nfs/kaiyuan/models/Qwen3-0.6B_torch_dist",
        help="Megatron checkpoint/model path",
    )
    parser.add_argument(
        "--data-path",
        type=str,
        default="/data/nfs/kaiyuan/datasets/dapo-math-17k/dapo-math-17k.jsonl",
        help="Training data path",
    )
    parser.add_argument("--output-dir", type=str, default="./megatron_out", help="Checkpoint output directory")
    parser.add_argument("--use-mock-data", action="store_true", help="Use Megatron fake/mock data")
    parser.add_argument(
        "--train-script",
        type=str,
        default="pretrain_gpt.py",
        help="Megatron training entry script (e.g. pretrain_gpt.py or train.py)",
    )
    parser.add_argument(
        "--train-extra-args",
        type=str,
        default="",
        help='Extra args appended to torchrun command, e.g. "--tensor-model-parallel-size 2 --seq-length 4096"',
    )
    parser.add_argument(
        "--env",
        type=str,
        nargs="*",
        default=[
            "CUDA_DEVICE_MAX_CONNECTIONS=1",
            "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True",
        ],
        help='Extra env vars, format KEY=VALUE, e.g. NCCL_DEBUG=INFO',
    )
    parser.add_argument("--detach", action="store_true", help="Start training and return immediately")
    parser.add_argument("--wait-timeout-s", type=int, default=0, help="If >0, wait training at most N seconds")
    return parser.parse_args()


def parse_env_kv(items: list[str]) -> dict[str, str]:
    out = {}
    for item in items or []:
        if "=" not in item:
            raise ValueError(f"Invalid env item: {item}, expected KEY=VALUE")
        k, v = item.split("=", 1)
        out[k] = v
    return out


def main():
    args = parse_args()
    init_kwargs = {"address": args.address} if args.address else {"num_cpus": args.num_cpus}
    ray.init(**init_kwargs)

    pg = None
    actor = None
    try:
        total_gpu = int(ray.cluster_resources().get("GPU", 0))
        if total_gpu < 8:
            raise RuntimeError(f"Need >=8 GPUs for this demo, found {total_gpu}")

        # 单bundle占满8卡，保证训练进程获得完整单机8GPU资源。
        pg = placement_group([{"CPU": 8, "GPU": 8}], strategy="PACK")
        print("waiting placement group ready...")
        ray.get(pg.ready())
        print("placement group ready.")

        sched = PlacementGroupSchedulingStrategy(
            placement_group=pg,
            placement_group_bundle_index=0,
            placement_group_capture_child_tasks=True,
        )
        actor = MegatronTrainActor.options(scheduling_strategy=sched).remote()
        print("actor info:", ray.get(actor.info.remote()))

        start_info = ray.get(
            actor.start.remote(
                train_script=args.train_script,
                model_path=args.model_path,
                data_path=args.data_path,
                output_dir=args.output_dir,
                nproc_per_node=args.nproc_per_node,
                use_mock_data=args.use_mock_data,
                train_extra_args=shlex.split(args.train_extra_args) if args.train_extra_args else [],
                env_overrides=parse_env_kv(args.env),
            )
        )
        print("start:", start_info)

        if args.detach:
            print("detach=True, training is running in background actor.")
            return

        wait_info = ray.get(actor.wait.remote(args.wait_timeout_s))
        print("wait:", wait_info)
    finally:
        # 非detach模式做清理；detach模式保留训练进程。
        if not getattr(args, "detach", False) and actor is not None:
            try:
                ray.get(actor.stop.remote())
            except Exception:
                pass
        if pg is not None and not getattr(args, "detach", False):
            remove_placement_group(pg)
        if not getattr(args, "detach", False):
            ray.shutdown()
        else:
            print("Skip cleanup due to detach=True. Remember to stop actor/pg manually.")
            time.sleep(0.2)


if __name__ == "__main__":
    main()



## Case5 Offload Training


一种offload的核心思路：

- 训练阶段：模型驻留在GPU上。
- Offload 阶段：将模型迁移到CPU，并释放GPU缓存。
- 恢复阶段：将模型迁回GPU。

启动方式：

```
python ray_offload_train_state_demo.py --sleep-s 10
```


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-


import argparse
import os
import socket
import time

import ray
import torch
from ray.util.placement_group import placement_group, remove_placement_group
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy


def _mb(x: int) -> float:
    return x / (1024 * 1024)


@ray.remote(num_cpus=4, num_gpus=1)
class TrainOffloadStateActor:
    def __init__(self, hidden_size: int = 4096, num_layers: int = 12):
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.device = "cpu"
        self.model = self._build_model().cpu()
        self.optim = torch.optim.AdamW(self.model.parameters(), lr=1e-4)

    def _build_model(self):
        layers = []
        for _ in range(self.num_layers):
            layers.append(torch.nn.Linear(self.hidden_size, self.hidden_size, bias=False))
            layers.append(torch.nn.GELU())
        return torch.nn.Sequential(*layers)

    def _gpu_mem(self):
        if not torch.cuda.is_available():
            return {
                "cuda_available": False,
                "allocated_mb": 0.0,
                "reserved_mb": 0.0,
                "free_mb": 0.0,
                "total_mb": 0.0,
            }
        dev = torch.cuda.current_device()
        free, total = torch.cuda.mem_get_info(dev)
        return {
            "cuda_available": True,
            "allocated_mb": round(_mb(torch.cuda.memory_allocated(dev)), 2),
            "reserved_mb": round(_mb(torch.cuda.memory_reserved(dev)), 2),
            "free_mb": round(_mb(free), 2),
            "total_mb": round(_mb(total), 2),
        }

    def info(self):
        return {
            "node": socket.gethostname(),
            "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
            "device_state": self.device,
            "gpu_mem": self._gpu_mem(),
        }

    def enter_train_phase(self):
        if torch.cuda.is_available():
            self.model = self.model.cuda()
            self.device = "cuda"
            torch.cuda.synchronize()
        return {"phase": "train", "device_state": self.device, "gpu_mem": self._gpu_mem()}

    def train_one_step(self, batch_size: int = 2):
        if self.device != "cuda":
            return {"ok": False, "reason": "model is not on cuda", "gpu_mem": self._gpu_mem()}
        x = torch.randn(batch_size, self.hidden_size, device="cuda")
        y = self.model(x).sum()
        self.optim.zero_grad(set_to_none=True)
        y.backward()
        self.optim.step()
        torch.cuda.synchronize()
        return {"ok": True, "phase": "train_step", "gpu_mem": self._gpu_mem()}

    def offload_train(self):
        # Mimic offload-train: move train model state out of GPU.
        self.model = self.model.cpu()
        self.device = "cpu"
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        return {"phase": "offload_train", "device_state": self.device, "gpu_mem": self._gpu_mem()}

    def resume_train(self):
        if torch.cuda.is_available():
            self.model = self.model.cuda()
            self.device = "cuda"
            torch.cuda.synchronize()
        return {"phase": "resume_train", "device_state": self.device, "gpu_mem": self._gpu_mem()}


def parse_args():
    p = argparse.ArgumentParser(description="Ray PG offload-train state demo")
    p.add_argument("--address", type=str, default=None, help="Ray address; empty=local")
    p.add_argument("--num-cpus", type=int, default=8, help="CPUs for local ray.init")
    p.add_argument("--sleep-s", type=float, default=1.0, help="Pause between phases")
    p.add_argument("--hidden-size", type=int, default=4096)
    p.add_argument("--num-layers", type=int, default=12)
    return p.parse_args()


def main():
    args = parse_args()
    init_kwargs = {"address": args.address} if args.address else {"num_cpus": args.num_cpus}
    ray.init(**init_kwargs)

    pg = None
    actor = None
    try:
        total_gpu = int(ray.cluster_resources().get("GPU", 0))
        if total_gpu < 1:
            raise RuntimeError(f"Need >=1 GPU, found {total_gpu}")

        pg = placement_group([{"CPU": 4, "GPU": 1}], strategy="PACK")
        print("waiting placement group ready...")
        ray.get(pg.ready())
        print("placement group ready.")

        sched = PlacementGroupSchedulingStrategy(
            placement_group=pg,
            placement_group_bundle_index=0,
            placement_group_capture_child_tasks=True,
        )
        actor = TrainOffloadStateActor.options(scheduling_strategy=sched).remote(
            hidden_size=args.hidden_size, num_layers=args.num_layers
        )

        print("initial:", ray.get(actor.info.remote()))
        print("enter_train_phase:", ray.get(actor.enter_train_phase.remote()))
        print("train_one_step:", ray.get(actor.train_one_step.remote()))
        time.sleep(args.sleep_s)

        print("offload_train:", ray.get(actor.offload_train.remote()))
        time.sleep(args.sleep_s)

        print("resume_train:", ray.get(actor.resume_train.remote()))
        time.sleep(args.sleep_s)
        print("train_one_step_again:", ray.get(actor.train_one_step.remote()))
    finally:
        if pg is not None:
            remove_placement_group(pg)
        ray.shutdown()


if __name__ == "__main__":
    main()



waiting placement group ready...
placement group ready.
initial: {'node': 'DevServer-BMS-292db97e', 'cuda_visible_devices': '0', 'device_state': 'cpu', 'gpu_mem': {'cuda_available': True, 'allocated_mb': 0.0, 'reserved_mb': 0.0, 'free_mb': 15848.5, 'total_mb': 81153.75}}
enter_train_phase: {'phase': 'train', 'device_state': 'cuda', 'gpu_mem': {'cuda_available': True, 'allocated_mb': 768.0, 'reserved_mb': 768.0, 'free_mb': 15080.5, 'total_mb': 81153.75}}
train_one_step: {'ok': True, 'phase': 'train_step', 'gpu_mem': {'cuda_available': True, 'allocated_mb': 3088.28, 'reserved_mb': 3862.0, 'free_mb': 11892.5, 'total_mb': 81153.75}}
offload_train: {'phase': 'offload_train', 'device_state': 'cpu', 'gpu_mem': {'cuda_available': True, 'allocated_mb': 1552.25, 'reserved_mb': 1556.0, 'free_mb': 14198.5, 'total_mb': 81153.75}}
resume_train: {'phase': 'resume_train', 'device_state': 'cuda', 'gpu_mem': {'cuda_available': True, 'allocated_mb': 3088.25, 'reserved_mb': 3092.0, 'free_mb': 12662.5, 't

## Case6 torch_memory_saver释放GPU显存

训练状态切换示例。

这个脚本用于演示训练过程中“训练 -> offload -> 恢复训练”的基本状态切换：
1) 训练阶段：模型常驻GPU，执行一步train_step。
2) Offload阶段：调用torch_memory_saver.pause()，释放托管显存。
3) 恢复阶段：调用torch_memory_saver.resume()，恢复训练所需显存。



启动参数

```
python ray_pg_megatron_pretrain_inproc_tms_demo.py   --tms-preload-so /usr/local/lib/python3.12/dist-packages/torch_memory_saver_hook_mode_preload.abi3.so
```

so包查找方式:

```
import os, torch_memory_saver
print(torch_memory_saver.__file__)
print(os.path.join(os.path.dirname(os.path.dirname(torch_memory_saver.__file__)),
                   "torch_memory_saver_hook_mode_preload.abi3.so"))
```

In [ ]:
# -*- coding: gbk -*-
#!/usr/bin/env python


import argparse
import os
import random
import socket
import time
from datetime import timedelta

import ray
import torch
import torch.distributed as dist
from ray.util.placement_group import placement_group, remove_placement_group
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy


def _mb(x: int) -> float:
    return round(x / (1024 * 1024), 2)


def _gpu_mem():
    if not torch.cuda.is_available():
        return {"cuda_available": False}
    dev = torch.cuda.current_device()
    free, total = torch.cuda.mem_get_info(dev)
    return {
        "cuda_available": True,
        "allocated_mb": _mb(torch.cuda.memory_allocated(dev)),
        "reserved_mb": _mb(torch.cuda.memory_reserved(dev)),
        "free_mb": _mb(free),
        "total_mb": _mb(total),
    }


@ray.remote(num_cpus=1, num_gpus=1)
class InprocTrainWorker:
    def __init__(self, world_size: int, rank: int, master_addr: str, master_port: int):
        self.world_size = world_size
        self.rank = rank
        self.master_addr = master_addr
        self.master_port = master_port
        self.model = None
        self.optim = None
        self._paused = False

        os.environ["MASTER_ADDR"] = str(master_addr)
        os.environ["MASTER_PORT"] = str(master_port)
        os.environ["WORLD_SIZE"] = str(world_size)
        os.environ["RANK"] = str(rank)
        os.environ["LOCAL_RANK"] = "0"
        self.pg_initialized = False
        self.graph_mode = False
        self.g = None
        self.static_x = None
        self.static_y = None
        self.vram_pad = None

    def _init_pg(self):
        if self.pg_initialized:
            return
        try:
            dist.init_process_group(
                backend="nccl",
                rank=self.rank,
                world_size=self.world_size,
                timeout=timedelta(seconds=90),
                device_id=torch.device("cuda", 0),
            )
        except TypeError:
            # Compatibility with older torch versions without device_id arg.
            dist.init_process_group(
                backend="nccl",
                rank=self.rank,
                world_size=self.world_size,
                timeout=timedelta(seconds=90),
            )
        self.pg_initialized = True

    def _destroy_pg(self):
        if not self.pg_initialized:
            return
        try:
            if dist.is_initialized():
                dist.destroy_process_group()
        finally:
            self.pg_initialized = False

    def init_runtime(
        self,
        hidden_size: int = 8192,
        num_layers: int = 40,
        batch_size: int = 4,
        target_vram_gb: float = 50.0,
        vram_headroom_gb: float = 4.0,
        graph_mode: bool = False,
    ):
        import torch_memory_saver

        # Compatibility: some versions expose pause/resume at module level,
        # while others expose an object `torch_memory_saver`.
        self.tms = (
            torch_memory_saver
            if hasattr(torch_memory_saver, "pause")
            else getattr(torch_memory_saver, "torch_memory_saver")
        )
        if not hasattr(self.tms, "pause") or not hasattr(self.tms, "resume"):
            raise RuntimeError(
                "torch_memory_saver API not found: expected pause()/resume() "
                "either on module or on torch_memory_saver.torch_memory_saver"
            )
        self.hidden_size = hidden_size
        self.batch_size = batch_size
        self.graph_mode = graph_mode
        torch.cuda.set_device(0)
        self._init_pg()

        layers = []
        for _ in range(num_layers):
            layers.append(torch.nn.Linear(hidden_size, hidden_size, bias=False, dtype=torch.bfloat16))
            layers.append(torch.nn.GELU())
        self.model = torch.nn.Sequential(*layers).cuda()
        self.optim = torch.optim.AdamW(self.model.parameters(), lr=1e-4, foreach=False, fused=False)
        if self.graph_mode:
            self.static_x = torch.empty(batch_size, hidden_size, device="cuda", dtype=torch.bfloat16)
            self.static_x.normal_(mean=0.0, std=1.0)

            # Warmup on a side stream (recommended for CUDA graph capture stability).
            warmup_stream = torch.cuda.Stream()
            warmup_stream.wait_stream(torch.cuda.current_stream())
            with torch.cuda.stream(warmup_stream):
                for _ in range(3):
                    self.optim.zero_grad(set_to_none=True)
                    y = self.model(self.static_x).sum()
                    y.backward()
                    self.optim.step()
            torch.cuda.current_stream().wait_stream(warmup_stream)
            torch.cuda.synchronize()

            # Capture graph for fwd+bwd. (optimizer step runs outside graph for robustness)
            self.g = torch.cuda.CUDAGraph()
            self.optim.zero_grad(set_to_none=True)
            with torch.cuda.graph(self.g):
                self.static_y = self.model(self.static_x).sum()
                self.static_y.backward()

        # Allocate extra persistent buffer to hit target VRAM usage.
        if torch.cuda.is_available():
            free_b, total_b = torch.cuda.mem_get_info(0)
            allocated_b = torch.cuda.memory_allocated(0)
            target_b = int(target_vram_gb * 1024**3)
            headroom_b = int(vram_headroom_gb * 1024**3)
            max_safe_b = max(0, total_b - headroom_b)
            want_b = min(target_b, max_safe_b)
            need_b = max(0, want_b - allocated_b)
            # bf16 buffer: 2 bytes per element
            n_elem = need_b // 2
            self.vram_pad = torch.empty(n_elem, dtype=torch.bfloat16, device="cuda") if n_elem > 0 else None

        torch.cuda.synchronize()
        return {
            "rank": self.rank,
            "phase": "init_done",
            "graph_mode": self.graph_mode,
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "batch_size": batch_size,
            "target_vram_gb": target_vram_gb,
            "gpu_mem": _gpu_mem(),
        }

    def train_step(self):
        if self._paused:
            return {"rank": self.rank, "ok": False, "reason": "paused", "gpu_mem": _gpu_mem()}
        if not self.pg_initialized:
            return {"rank": self.rank, "ok": False, "reason": "pg_not_initialized", "gpu_mem": _gpu_mem()}

        if self.graph_mode:
            # fake data: refresh static input in-place then replay graph
            self.static_x.normal_(mean=0.0, std=1.0)
            self.optim.zero_grad(set_to_none=True)
            self.g.replay()
            self.optim.step()
            self.optim.zero_grad(set_to_none=True)
            phase = "train_step_cuda_graph"
        else:
            # eager fallback for better compatibility with torch_memory_saver pause/resume
            x = torch.randn(self.batch_size, self.hidden_size, device="cuda", dtype=torch.bfloat16)
            y = self.model(x).sum()
            self.optim.zero_grad(set_to_none=True)
            y.backward()
            self.optim.step()
            self.optim.zero_grad(set_to_none=True)
            phase = "train_step_eager"
        dist.barrier()
        torch.cuda.synchronize()
        return {
            "rank": self.rank,
            "ok": True,
            "phase": phase,
            "gpu_mem": _gpu_mem(),
        }

    def sleep(self):
        # barrier -> destroy process groups -> pause memory saver.
        torch.cuda.synchronize()
        if self.pg_initialized and dist.is_initialized():
            dist.barrier()
        self._destroy_pg()
        # Release graph-related and pad buffers before tms.pause(),
        # which improves compatibility with some tms/cuda versions.
        self.g = None
        self.static_x = None
        self.static_y = None
        self.vram_pad = None
        torch.cuda.empty_cache()
        self.tms.pause()
        self._paused = True
        return {"rank": self.rank, "phase": "sleep", "gpu_mem": _gpu_mem()}

    def wake_up(self):
        self.tms.resume()
        self._paused = False
        torch.cuda.synchronize()
        return {"rank": self.rank, "phase": "wake_up_resume_only", "gpu_mem": _gpu_mem()}

    def reinit_pg(self, master_addr: str, master_port: int):
        # Rebuild process groups with a fresh master port to avoid stale rendezvous.
        os.environ["MASTER_ADDR"] = str(master_addr)
        os.environ["MASTER_PORT"] = str(master_port)
        self.master_addr = master_addr
        self.master_port = master_port
        self._init_pg()
        return {
            "rank": self.rank,
            "phase": "reinit_pg",
            "master_addr": self.master_addr,
            "master_port": self.master_port,
            "gpu_mem": _gpu_mem(),
        }

    def shutdown(self):
        try:
            if self.pg_initialized and dist.is_initialized():
                dist.barrier()
        except Exception:
            pass
        self._destroy_pg()
        return {"rank": self.rank, "phase": "shutdown"}

    def info(self):
        return {
            "rank": self.rank,
            "node": socket.gethostname(),
            "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
            "ld_preload": os.environ.get("LD_PRELOAD"),
            "paused": self._paused,
            "gpu_mem": _gpu_mem(),
        }


def parse_args():
    p = argparse.ArgumentParser(description="In-process Ray PG Megatron-style pause/resume demo")
    p.add_argument("--address", type=str, default=None)
    p.add_argument("--num-cpus", type=int, default=16)
    p.add_argument("--world-size", type=int, default=8, help="single-node 8 GPU demo")
    p.add_argument("--hidden-size", type=int, default=8192)
    p.add_argument("--num-layers", type=int, default=40)
    p.add_argument("--batch-size", type=int, default=4)
    p.add_argument(
        "--graph-mode",
        action="store_true",
        help="Enable CUDA Graph mode. Default off for better tms pause compatibility.",
    )
    p.add_argument("--target-vram-gb", type=float, default=50.0, help="Target per-GPU memory occupancy")
    p.add_argument("--vram-headroom-gb", type=float, default=4.0, help="Keep this many GB free to avoid OOM")
    p.add_argument(
        "--tms-preload-so",
        type=str,
        default="/usr/local/lib/python3.12/dist-packages/torch_memory_saver_hook_mode_preload.abi3.so",
    )
    return p.parse_args()


def main():
    args = parse_args()
    init_kwargs = {"address": args.address} if args.address else {"num_cpus": args.num_cpus}
    ray.init(**init_kwargs)

    pg = None
    workers = []
    try:
        total_gpu = int(ray.cluster_resources().get("GPU", 0))
        if total_gpu < args.world_size:
            raise RuntimeError(f"Need >= {args.world_size} GPUs, found {total_gpu}")

        pg = placement_group([{"CPU": 1, "GPU": 1} for _ in range(args.world_size)], strategy="PACK")
        print("waiting placement group ready...")
        ray.get(pg.ready())
        print("placement group ready.")

        master_addr = ray.util.get_node_ip_address()
        master_port = random.randint(20000, 21000)
        env_vars = {
            "LD_PRELOAD": args.tms_preload_so,
            "TMS_INIT_ENABLE": "1",
            "TMS_INIT_ENABLE_CPU_BACKUP": "1",
            "CUDA_DEVICE_MAX_CONNECTIONS": "1",
        }

        for rank in range(args.world_size):
            sched = PlacementGroupSchedulingStrategy(
                placement_group=pg,
                placement_group_bundle_index=rank,
                placement_group_capture_child_tasks=True,
            )
            w = InprocTrainWorker.options(
                scheduling_strategy=sched,
                runtime_env={"env_vars": env_vars},
            ).remote(
                world_size=args.world_size,
                rank=rank,
                master_addr=master_addr,
                master_port=master_port,
            )
            workers.append(w)

        print("workers init...")
        init_out = ray.get(
            [
                w.init_runtime.remote(
                    args.hidden_size,
                    args.num_layers,
                    args.batch_size,
                    args.target_vram_gb,
                    args.vram_headroom_gb,
                    args.graph_mode,
                )
                for w in workers
            ]
        )
        print("init sample:", init_out[0])

        print("train step #1 ...")
        out1 = ray.get([w.train_step.remote() for w in workers])
        print("rank0:", out1[0])

        print("sleep() all workers ...")
        sleep_out = ray.get([w.sleep.remote() for w in workers])
        print("rank0:", sleep_out[0])

        print("wake_up() all workers ...")
        wake_out = ray.get([w.wake_up.remote() for w in workers])
        print("rank0 resume:", wake_out[0])
        # Re-init process groups with a NEW port after resume.
        new_master_port = random.randint(21001, 22000)
        reinit_out = ray.get([w.reinit_pg.remote(master_addr, new_master_port) for w in workers])
        print("rank0 reinit:", reinit_out[0])

        print("train step #2 ...")
        out2 = ray.get([w.train_step.remote() for w in workers])
        print("rank0:", out2[0])

    finally:
        # Coordinated shutdown with timeout to avoid hanging forever.
        if workers:
            shutdown_refs = [w.shutdown.remote() for w in workers]
            try:
                ray.get(shutdown_refs, timeout=30)
            except Exception:
                for w in workers:
                    try:
                        ray.kill(w)
                    except Exception:
                        pass
        if pg is not None:
            remove_placement_group(pg)
        ray.shutdown()


if __name__ == "__main__":
    main()



## Case7 训练推理共卡，交替使用资源

演示在同一组GPU资源上交替执行训练与推理：
- 训练侧使用Ray worker + torch_memory_saver进行sleep/wake切换。
- 推理侧使用SGLang服务进行sleep/resume切换。
- 通过两次/generate请求验证推理在启动后和resume后都可用。

执行步骤：
1) 创建共享Placement Group并探测bundle到物理GPU映射。
2) 启动训练worker，完成初始化和第1次train_step。
3) 训练进入sleep（tms.pause），释放训练占用。
4) 启动SGLang并等待健康检查，通过第1次/generate验证可用。
5) SGLang进入sleep_mode（release_memory_occupation）。
6) 训练wake_up后执行第2次train_step，并再次sleep。
7) SGLang resume_mode（resume_memory_occupation），做health与第2次/generate验证。



需要配置的主要参数：

--tms-preload-so type=str, default="/usr/local/lib/python3.12/dist-packages/torch_memory_saver_hook_mode_preload.abi3.so"

--sglang-model-path type=str, default="/data/nfs/kaiyuan/models/Qwen3-8B"

执行
```
python run.py ----tms-preload-so xxxx --sglang-model-path xxx
```

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
New standalone demo: train/infer interleaving with Ray PG.

Flow:
1) Train workers init + train step
2) Train sleep (tms pause)
3) Start SGLang service
4) SGLang sleep mode (release_memory_occupation)
5) Train wake_up + train step + sleep
6) SGLang resume (resume_memory_occupation)
7) Health check via curl
"""

import argparse
import multiprocessing
import os
import random
import socket
import subprocess
import time
from datetime import timedelta

import ray
import requests
import torch
import torch.distributed as dist
from ray.util.placement_group import placement_group, remove_placement_group
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy
from sglang.srt.entrypoints.http_server import launch_server
from sglang.srt.server_args import ServerArgs

NOSET_VISIBLE_DEVICES_ENV_VARS = {
    "RAY_EXPERIMENTAL_NOSET_CUDA_VISIBLE_DEVICES": "1",
    "RAY_EXPERIMENTAL_NOSET_ROCR_VISIBLE_DEVICES": "1",
    "RAY_EXPERIMENTAL_NOSET_ASCEND_RT_VISIBLE_DEVICES": "1",
    "RAY_EXPERIMENTAL_NOSET_HABANA_VISIBLE_MODULES": "1",
    "RAY_EXPERIMENTAL_NOSET_NEURON_RT_VISIBLE_CORES": "1",
    "RAY_EXPERIMENTAL_NOSET_TPU_VISIBLE_CHIPS": "1",
    "RAY_EXPERIMENTAL_NOSET_ONEAPI_DEVICE_SELECTOR": "1",
}


def _mb(x: int) -> float:
    return round(x / (1024 * 1024), 2)


def _gpu_mem():
    if not torch.cuda.is_available():
        return {"cuda_available": False}
    dev = torch.cuda.current_device()
    free, total = torch.cuda.mem_get_info(dev)
    return {
        "cuda_available": True,
        "allocated_mb": _mb(torch.cuda.memory_allocated(dev)),
        "reserved_mb": _mb(torch.cuda.memory_reserved(dev)),
        "free_mb": _mb(free),
        "total_mb": _mb(total),
    }


def _curl_generate(host: str, port: int, question: str):
    """使用curl调用/generate做一次简单问答，用于验证SGLang推理是否可用。"""
    generate_url = f"http://{host}:{port}/generate"
    payload = (
        '{"text": "'
        + question.replace('"', '\\"')
        + '", "sampling_params": {"max_new_tokens": 32}}'
    )
    cmd = [
        "curl",
        "-sS",
        "-X",
        "POST",
        "-H",
        "Content-Type: application/json",
        "-d",
        payload,
        generate_url,
    ]
    print("curl generate:", " ".join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    print("curl generate stdout:", res.stdout.strip())
    if res.stderr:
        print("curl generate stderr:", res.stderr.strip())


@ray.remote(num_cpus=1, num_gpus=1)
class TrainWorker:
    def __init__(self, world_size: int, rank: int, master_addr: str, master_port: int):
        self.world_size = world_size
        self.rank = rank
        self.master_addr = master_addr
        self.master_port = master_port
        self.pg_initialized = False
        self._paused = False
        self.model = None
        self.optim = None

        os.environ["MASTER_ADDR"] = str(master_addr)
        os.environ["MASTER_PORT"] = str(master_port)
        os.environ["WORLD_SIZE"] = str(world_size)
        os.environ["RANK"] = str(rank)
        os.environ["LOCAL_RANK"] = "0"

    def _init_pg(self):
        if self.pg_initialized:
            return
        dist.init_process_group(
            backend="nccl",
            rank=self.rank,
            world_size=self.world_size,
            timeout=timedelta(seconds=90),
        )
        self.pg_initialized = True

    def _destroy_pg(self):
        if self.pg_initialized and dist.is_initialized():
            dist.destroy_process_group()
        self.pg_initialized = False

    def init_runtime(self, hidden_size: int = 4096, num_layers: int = 12):
        import torch_memory_saver

        self.tms = (
            torch_memory_saver
            if hasattr(torch_memory_saver, "pause")
            else getattr(torch_memory_saver, "torch_memory_saver")
        )
        torch.cuda.set_device(0)
        self._init_pg()

        layers = []
        for _ in range(num_layers):
            layers.append(torch.nn.Linear(hidden_size, hidden_size, bias=False, dtype=torch.bfloat16))
            layers.append(torch.nn.GELU())
        self.model = torch.nn.Sequential(*layers).cuda()
        self.optim = torch.optim.AdamW(self.model.parameters(), lr=1e-4, foreach=False, fused=False)
        self.hidden_size = hidden_size
        return {"rank": self.rank, "phase": "init_done", "gpu_mem": _gpu_mem()}

    def train_step(self, batch_size: int = 2):
        if self._paused:
            return {"rank": self.rank, "ok": False, "reason": "paused", "gpu_mem": _gpu_mem()}
        x = torch.randn(batch_size, self.hidden_size, device="cuda", dtype=torch.bfloat16)
        y = self.model(x).sum()
        self.optim.zero_grad(set_to_none=True)
        y.backward()
        self.optim.step()
        self.optim.zero_grad(set_to_none=True)
        if self.pg_initialized and dist.is_initialized():
            dist.barrier()
        torch.cuda.synchronize()
        return {"rank": self.rank, "phase": "train_step", "gpu_mem": _gpu_mem()}

    def sleep(self):
        if self.pg_initialized and dist.is_initialized():
            dist.barrier()
        self._destroy_pg()
        torch.cuda.empty_cache()
        self.tms.pause()
        self._paused = True
        return {"rank": self.rank, "phase": "sleep", "gpu_mem": _gpu_mem()}

    def wake_up(self, master_addr: str, master_port: int):
        self.tms.resume()
        os.environ["MASTER_ADDR"] = str(master_addr)
        os.environ["MASTER_PORT"] = str(master_port)
        self.master_addr = master_addr
        self.master_port = master_port
        self._init_pg()
        self._paused = False
        return {"rank": self.rank, "phase": "wake_up", "gpu_mem": _gpu_mem()}

    def shutdown(self):
        self._destroy_pg()
        return {"rank": self.rank, "phase": "shutdown"}


@ray.remote(num_cpus=1, num_gpus=0.01)
class BundleProbe:
    def probe(self):
        gpu_ids = ray.get_gpu_ids()
        gid = int(gpu_ids[0]) if gpu_ids else -1
        return {"node": socket.gethostname(), "gpu_id": gid, "all_gpu_ids": gpu_ids}


def _wait_server_healthy(base_url: str, timeout_s: int = 180, status_cb=None):
    deadline = time.time() + timeout_s
    last = None
    attempt = 0
    while time.time() < deadline:
        attempt += 1
        try:
            r = requests.get(f"{base_url}/health_generate", timeout=2)
            if r.status_code in (200, 503):
                return
            last = f"status={r.status_code}"
        except Exception as e:
            last = str(e)
        if status_cb is not None and (attempt == 1 or attempt % 10 == 0):
            status = status_cb()
            print(f"[SGLangHealth] attempt={attempt} last={last} status={status}", flush=True)
        time.sleep(2)
    raise TimeoutError(f"SGLang health check timeout: {last}")


def _launch_sglang(server_args: ServerArgs) -> multiprocessing.Process:
    multiprocessing.set_start_method("spawn", force=True)
    server_args.host = server_args.host.strip("[]")
    p = multiprocessing.Process(target=launch_server, args=(server_args,))
    p.start()
    return p


@ray.remote(num_cpus=1, num_gpus=2)
class SGLangActor:
    def __init__(self, model_path: str, host: str, port: int, tp_size: int):
        self.model_path = model_path
        self.host = host
        self.port = port
        self.tp_size = tp_size
        self.proc = None

    def start(self, gpu_ids: list[int] | None = None):
        if self.proc is not None and self.proc.is_alive():
            return {"status": "already_running", "pid": self.proc.pid}
        if not gpu_ids or len(gpu_ids) < self.tp_size:
            raise RuntimeError(f"need at least {self.tp_size} gpu_ids, got {gpu_ids}")

        os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(x) for x in gpu_ids[: self.tp_size])
        print(
            "[SGLangActor.start] "
            f"ray_gpu_ids={ray.get_gpu_ids()} "
            f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}",
            flush=True,
        )
        args = ServerArgs(
            model_path=self.model_path,
            host=self.host,
            port=self.port,
            tp_size=self.tp_size,
            nnodes=1,
            node_rank=0,
            dist_init_addr=f"127.0.0.1:{self.port+1}",
            nccl_port=self.port + 2,
            trust_remote_code=True,
        )
        self.proc = _launch_sglang(args)
        return {
            "status": "started",
            "pid": self.proc.pid,
            "base_url": f"http://{self.host}:{self.port}",
            "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
            "ray_gpu_ids": ray.get_gpu_ids(),
        }

    def wait_until_healthy(self, timeout_s: int = 180):
        _wait_server_healthy(
            f"http://{self.host}:{self.port}",
            timeout_s=timeout_s,
            status_cb=lambda: {
                "proc_alive": (self.proc.is_alive() if self.proc is not None else False),
                "proc_exitcode": (self.proc.exitcode if self.proc is not None else None),
                "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
                "ray_gpu_ids": ray.get_gpu_ids(),
            },
        )
        if self.proc is not None and not self.proc.is_alive():
            raise RuntimeError(f"SGLang exited early, exitcode={self.proc.exitcode}")
        return {"status": "healthy", "url": f"http://{self.host}:{self.port}/health_generate"}

    def flush_cache(self, max_retries: int = 60):
        """Best-effort/flush_cache"""
        url = f"http://{self.host}:{self.port}/flush_cache"
        last_err = None
        for _ in range(max_retries):
            try:
                r = requests.get(url, timeout=10)
                if r.status_code == 200:
                    return {"status": "flush_ok"}
                last_err = f"status={r.status_code}, body={r.text[:300]}"
            except Exception as e:
                last_err = str(e)
            time.sleep(1.0)
        raise TimeoutError(f"Timeout while flushing cache: {last_err}")

    def sleep_mode(self):
        # 先flush_cache，再做offload
        self.flush_cache()
        url = f"http://{self.host}:{self.port}/release_memory_occupation"
        last_err = None
        # SGLang要求release_memory_occupation只能在“无在途请求”时调用。
        # 刚做完health请求后可能短暂仍有请求在处理，因此这里做短重试。
        for _ in range(20):
            try:
                r = requests.post(url, json={}, timeout=10)
                if r.status_code == 200:
                    return {"status": "sglang_sleep", "url": url, "resp": r.json() if r.content else {}}
                last_err = f"status={r.status_code}, body={r.text[:300]}"
            except Exception as e:
                last_err = str(e)
            time.sleep(1.0)
        raise RuntimeError(f"failed to enter sglang sleep mode after retries: {last_err}")

    def resume_mode(self):
        url = f"http://{self.host}:{self.port}/resume_memory_occupation"
        r = requests.post(url, json={"tags": None}, timeout=10)
        r.raise_for_status()
        return {"status": "sglang_resume", "url": url, "resp": r.json() if r.content else {}}

    def stop(self):
        if self.proc is None:
            return {"status": "not_started"}
        if self.proc.is_alive():
            self.proc.terminate()
            try:
                self.proc.join(timeout=15)
            except Exception:
                self.proc.kill()
        return {"status": "stopped", "exitcode": self.proc.exitcode}


def parse_args():
    p = argparse.ArgumentParser(description="Ray train/infer interleave demo")
    p.add_argument("--address", type=str, default=None)
    p.add_argument("--num-cpus", type=int, default=16)
    p.add_argument("--world-size", type=int, default=8)
    p.add_argument(
        "--train-world-size",
        type=int,
        default=8,
        help="number of train workers; set 8 for full colocate-style training",
    )
    p.add_argument("--train-gpu-fraction", type=float, default=0.4, help="GPU fraction per train worker in shared PG")
    p.add_argument("--sglang-gpu-fraction", type=float, default=0.2, help="GPU fraction for sglang actor in shared PG")
    p.add_argument(
        "--sglang-bundle-indices",
        type=str,
        default="0,1",
        help="bundle indices for sglang tp workers in shared PG, e.g. '0,1'",
    )
    p.add_argument("--train-hidden-size", type=int, default=4096)
    p.add_argument("--train-num-layers", type=int, default=12)
    p.add_argument("--tms-preload-so", type=str, default="/usr/local/lib/python3.12/dist-packages/torch_memory_saver_hook_mode_preload.abi3.so")
    p.add_argument("--sglang-model-path", type=str, default="/data/nfs/kaiyuan/models/Qwen3-8B")
    p.add_argument("--sglang-host", type=str, default="127.0.0.1")
    p.add_argument("--sglang-port", type=int, default=30000)
    p.add_argument("--sglang-tp-size", type=int, default=2)
    p.add_argument("--sglang-health-timeout-s", type=int, default=45, help="SGLang health timeout seconds")
    return p.parse_args()


def main():
    args = parse_args()
    init_kwargs = {"address": args.address} if args.address else {"num_cpus": args.num_cpus}
    ray.init(**init_kwargs)

    pg_shared = None
    workers = []
    sgl = None
    try:
        # ---------- Shared PG for co-card ----------
        pg_shared = placement_group([{"CPU": 2, "GPU": 1} for _ in range(args.world_size)], strategy="PACK")
        print("waiting shared PG ready...")
        ray.get(pg_shared.ready())
        print("shared PG ready.")

        # Probe bundle -> physical GPU mapping
        bundle_gpu_ids = []
        for i in range(args.world_size):
            sched = PlacementGroupSchedulingStrategy(
                placement_group=pg_shared,
                placement_group_bundle_index=i,
                placement_group_capture_child_tasks=True,
            )
            probe = BundleProbe.options(scheduling_strategy=sched).remote()
            info = ray.get(probe.probe.remote())
            bundle_gpu_ids.append(info["gpu_id"])
            ray.kill(probe)
        print("bundle->gpu:", bundle_gpu_ids)

        if args.train_world_size > args.world_size - args.sglang_tp_size:
            # colocate mode: allow full train_world_size if fractional resources can co-exist.
            if args.train_world_size != args.world_size:
                raise RuntimeError(
                    f"train_world_size={args.train_world_size} too large for world_size={args.world_size} "
                    f"with sglang_tp_size={args.sglang_tp_size}"
                )

        sglang_bundle_indices = [int(x.strip()) for x in args.sglang_bundle_indices.split(",") if x.strip()]
        if len(sglang_bundle_indices) != args.sglang_tp_size:
            raise RuntimeError(
                f"sglang_bundle_indices({sglang_bundle_indices}) size must equal sglang_tp_size={args.sglang_tp_size}"
            )
        for b in sglang_bundle_indices:
            if b < 0 or b >= args.world_size:
                raise RuntimeError(f"invalid sglang bundle index {b}, world_size={args.world_size}")

        # Resource feasibility check for colocated bundles.
        if args.train_world_size == args.world_size:
            if args.train_gpu_fraction + args.sglang_gpu_fraction > 1.0:
                raise RuntimeError(
                    "for full-colocate mode, train_gpu_fraction + sglang_gpu_fraction must be <= 1.0 "
                    f"(got {args.train_gpu_fraction} + {args.sglang_gpu_fraction})"
                )
            if args.sglang_gpu_fraction < 1e-6:
                raise RuntimeError("sglang_gpu_fraction must be > 0")

        train_bundle_indices = list(range(args.train_world_size))
        print("train bundle indices:", train_bundle_indices)
        print("sglang bundle indices:", sglang_bundle_indices)

        master_addr = ray.util.get_node_ip_address()
        master_port = random.randint(20000, 21000)
        train_env = {
            "LD_PRELOAD": args.tms_preload_so,
            "TMS_INIT_ENABLE": "1",
            "TMS_INIT_ENABLE_CPU_BACKUP": "1",
            "CUDA_DEVICE_MAX_CONNECTIONS": "1",
        }
        for rank, bundle_idx in enumerate(train_bundle_indices):
            sched = PlacementGroupSchedulingStrategy(
                placement_group=pg_shared,
                placement_group_bundle_index=bundle_idx,
                placement_group_capture_child_tasks=True,
            )
            w = TrainWorker.options(
                num_gpus=args.train_gpu_fraction,
                scheduling_strategy=sched,
                runtime_env={"env_vars": train_env},
            ).remote(args.train_world_size, rank, master_addr, master_port)
            workers.append(w)

        print("workers init...")
        init_out = ray.get([w.init_runtime.remote(args.train_hidden_size, args.train_num_layers) for w in workers])
        print("init sample:", init_out[0])

        print("train step #1 ...")
        out1 = ray.get([w.train_step.remote() for w in workers])
        print("rank0:", out1[0])

        print("train sleep ...")
        sleep_out = ray.get([w.sleep.remote() for w in workers])
        print("rank0:", sleep_out[0])

        # ---------- SGLang side (same shared PG, co-card) ----------
        sgl_sched = PlacementGroupSchedulingStrategy(
            placement_group=pg_shared,
            placement_group_bundle_index=sglang_bundle_indices[0],
            placement_group_capture_child_tasks=True,
        )
        sgl = SGLangActor.options(
            num_gpus=args.sglang_gpu_fraction,
            scheduling_strategy=sgl_sched,
            runtime_env={"env_vars": NOSET_VISIBLE_DEVICES_ENV_VARS},
        ).remote(
            args.sglang_model_path, args.sglang_host, args.sglang_port, args.sglang_tp_size
        )
        # pick gpu ids from reserved sglang bundles
        selected_gpu_ids = []
        for bidx in sglang_bundle_indices:
            gid = bundle_gpu_ids[bidx]
            if gid >= 0 and gid not in selected_gpu_ids:
                selected_gpu_ids.append(gid)
        print("sglang selected gpu ids:", selected_gpu_ids)
        print("sglang start:", ray.get(sgl.start.remote(selected_gpu_ids)))
        print(f"sglang wait healthy (timeout={args.sglang_health_timeout_s}s) ...")
        print("sglang health:", ray.get(sgl.wait_until_healthy.remote(args.sglang_health_timeout_s)))
        # 第一次生成请求：确认刚启动时推理可用
        _curl_generate(args.sglang_host, args.sglang_port, "Do you subscribe InfraTech?")

        print("sglang sleep mode ...")
        print("sglang sleep:", ray.get(sgl.sleep_mode.remote()))

        # ---------- Train resume ----------
        print("train wake_up ...")
        new_port = random.randint(21001, 22000)
        wake_out = ray.get([w.wake_up.remote(master_addr, new_port) for w in workers])
        print("rank0:", wake_out[0])

        print("train step #2 ...")
        out2 = ray.get([w.train_step.remote() for w in workers])
        print("rank0:", out2[0])

        print("train sleep again ...")
        sleep2 = ray.get([w.sleep.remote() for w in workers])
        print("rank0:", sleep2[0])

        # ---------- SGLang resume + curl health ----------
        print("sglang resume mode ...")
        print("sglang resume:", ray.get(sgl.resume_mode.remote()))

        # curl health check
        curl_cmd = [
            "curl",
            "-sS",
            "-o",
            "/dev/null",
            "-w",
            "%{http_code}",
            f"http://{args.sglang_host}:{args.sglang_port}/health_generate",
        ]
        curl_res = subprocess.run(curl_cmd, capture_output=True, text=True)
        print("curl health http_code:", curl_res.stdout.strip(), "stderr:", curl_res.stderr.strip())

        # 第二次生成请求：resume之后再确认一次推理可用
        _curl_generate(args.sglang_host, args.sglang_port, "Do you subscribe InfraTech?")

    finally:
        for w in workers:
            try:
                ray.get(w.shutdown.remote())
            except Exception:
                try:
                    ray.kill(w)
                except Exception:
                    pass
        if sgl is not None:
            try:
                ray.get(sgl.stop.remote())
            except Exception:
                pass
        if pg_shared is not None:
            remove_placement_group(pg_shared)
        ray.shutdown()


if __name__ == "__main__":
    main()

